# Prueba de funcionlidad `procesarOT.py`

### Cargar Librerias

In [1]:
import os
import sys
import time
from datetime import datetime
import pymongo
from pymongo.errors import ConnectionFailure
import logging

logging.basicConfig(level=logging.INFO)

from eerssa.secret import Keys

# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v30        # Coleccion actual
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


Success!!!


INFO:root::::: Conexion exitosa con MongoDB ::::


In [2]:
# DASK

from dask.distributed import LocalCluster, as_completed
dask = LocalCluster().get_client()
visor_dask = dask.dashboard_link

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42683 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.diskutils:Found stale lock file and directory '/tmp/dask-scratch-space/worker-e90ck9tf', purging
INFO:distributed.diskutils:Found stale lock file and directory '/tmp/dask-scratch-space/worker-ov8d_lnc', purging
INFO:distributed.diskutils:Found stale lock file and directory '/tmp/dask-scratch-space/worker-ce2lkphe', purging
INFO:distributed.diskutils:Found stale lock file and directory '/tmp/dask-scratch-space/scheduler-eg2nnjf2', purging
INFO:distributed.diskutils:Found stale lock file and directory '/tmp/dask-scratch-space/worker-k7b

In [3]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import procesarOt as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import generarMatrizActividades as Actividades     # process ot.data["actividades"]

### Recargar Librerias

In [15]:
reload( OrdenTrabajo )
reload( Actividades  )
print(visor_dask)

http://127.0.0.1:42683/status


### Directorios de Prueba

In [5]:
from pathlib import Path

dir_test = Path ("/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test")

# Get all entries (files and subdirectories)
all_entries = dir_test.iterdir()

# Filter for only files
files = [item for item in all_entries if item.is_file()]

# You can also get just the names if you prefer
# file_names = [item.name for item in all_entries if item.is_file()]

print("All files in the directory (Path objects):")
print(files)

# If you need them as strings
file_strings = [str(f) for f in files]
print("\nAll files as strings:")
print(file_strings)

All files in the directory (Path objects):
[PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/28_6 colaboradores.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/7_one_line_text_overlap.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_LM_Tres_hojas.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/12_casoEspecial01.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/14_Orden de trabajo Guayzimi 10-04-2022 (CQ).pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/27_5 colaboradores.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/8_bad_line_text_overlap.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/21_0 colaboradores y vehiculo.pdf'), PosixPath('/

### Test con una sola hoja

In [20]:
nro_ot_test =3
ot_test = OrdenTrabajo.procesarOt(file_strings[ nro_ot_test ])
ot_test

{'version': '0.3.0',
 'link': '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf',
 'exito': True,
 'log': [{'t': '2025-08-03T16:18:43.701080',
   'level': 'INFO',
   'message': 'CREACION DE LA OT, se encuentra un archivo PDF valido',
   'detail': 'Ubicacion: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf'}],
 'createdAt': '2025-08-03T16:18:43.701092',
 'id_ot': 81013,
 'diaSemana': 'viernes',
 'fecha': '2022-02-11T00:00:00-05:00',
 'trabajo': ['A - SIN VOLTAJE',
  'CON ENERGÍA/VOLTAJE - BAJO VOLTAJE',
  'CON ENERGÍA/VOLTAJE - MEDIO VOLTAJE',
  'TRABAJO EN ALTURA - SOBRE NIVEL',
  'DESCRIPCIÓN DEL TRABAJO A REALIZAR:'],
 'riesgos': ['BIOLÓGICOS',
  'Virus',
  'Infección viral',
  'Usar implementos de seguridad sanitaria (mascarilla, guantes, gel, alcohol)',
  'ELÉCTRICOS',
  'Descarga eléctrica',
  'Electrocución o electrización',
  

### DASK Parallel Computing

#### BAGS

In [7]:
import dask.bag as db
import dask
from datetime import datetime

file_strings

print(f"Se han encontrado un total de: {len(file_strings)} Ordenes de Trabajo")

start_time = time.time()
start_datetime = datetime.now()
print(f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n")

# 1. Create a Dask Bag from the list of PDF file paths
# Dask Bags are great for unstructured or custom data processing
dask_bag = db.from_sequence(file_strings)

# 2. Use the .map() method to apply the procesarOt function to each item
# This creates a computation graph. The function is not executed yet.
# `procesarOt` should be a standalone function, as it is in your provided code.
mapped_bag = dask_bag.map(OrdenTrabajo.procesarOt)

# 3. Call .compute() to trigger the parallel execution and get the final list of results.
# Dask will handle the scheduling of these tasks across workers.
obj_lists_dask = mapped_bag.compute()

end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")

Se han encontrado un total de: 28 Ordenes de Trabajo
Hora de inicio: 2025-08-03 15:20:53


Success!!!
Success!!!
Success!!!
Success!!!


   Procesados todos los 28 items. Tiempo transcurrido: 2.22 segundos.
   Hora Final : 2025-08-03 15:20:55


#### Futures

In [ ]:
# Verificar la conversion de archivos

print(f" Se han encontrado un total de: {len(file_strings)} Ordenes de Trabajo" )

start_time = time.time()
start_datetime = datetime.now()
print( f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n" )

# 1. Submit the first batch of tasks
# This returns a list of futures, same as before.
futures_step1 = [dask.submit(OrdenTrabajo.procesarOt, file, pure=False) for file in file_strings]

# 2. Submit the second batch of tasks, feeding the first futures as input
#futures_step2 = [dask.submit(call_load_ot, f) for f in futures_step1]

# 3. Now, gather only the FINAL results
# This single call executes the entire graph (both GestionOt and load_ot) in parallel.
obj_lists_dask = dask.gather(futures_step1)


end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")



In [78]:
dask.cancel(futures_step1)

In [ ]:
dask.close()

In [9]:
obj_lists_dask[0]

{'version': '0.3.0',
 'link': '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/28_6 colaboradores.pdf',
 'exito': True,
 'log': [{'t': '2025-08-03T15:20:54.735609',
   'level': 'INFO',
   'message': 'CREACION DE LA OT, se encuentra un archivo PDF valido',
   'detail': 'Ubicacion: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/28_6 colaboradores.pdf'},
  {'t': '2025-08-03T15:20:54.949598',
   'level': 'ERROR',
   'message': 'No coincide la FECHA OT: 2018-07-07T00:00:00-05:00 con la FECHA inicio: list index out of range',
   'detail': 'Revisar las fechas en la primer hoja.'}],
 'createdAt': '2025-08-03T15:20:54.735620',
 'diaSemana': 'sábado',
 'fecha': '2018-07-07T00:00:00-05:00',
 'fechaInicio': IndexError('list index out of range'),
 'trabajo': ['A - SIN VOLTAJE',
  'CON ENERGÍA/VOLTAJE - ALTO VOLTAJE',
  'CON ENERGÍA/VOLTAJE - BAJO VOLTAJE',
  'CON ENERGÍA/VOLTAJE - MEDIO VOLTAJE',
  'EN ÁREAS CONFINADOS - EXCAVACIÓN',
  'EN ÁREAS CONFINADOS - INTERIOR EQUIPO',
 